# 06 -- Result Tables (Main + Ablation) + Case Study

Two-table layout, standard ML-paper structure:
* **Table 1 (Main Results)**: non-trained baselines (vanilla GISMo, GISMo + post-hoc filter) vs our trained methods (v2 / v3 / v4 at best lambda).
* **Table 2 (Ablation Study)**: w/o each component vs Full model.
* **Lambda sweep panel**: Pareto curve (MRR <-> health satisfaction) across lambda values.
* **Case study**: qualitative samples showing how g flips predictions.

Edit `BEST_LAMBDA` and the path dicts under "Config" to point at your actual training outputs.

## 1. Environment

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q torch_geometric pandas numpy matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
PROJECT_ROOT = '/content/drive/MyDrive/CS471_project'
CODE_DIR = f'{PROJECT_ROOT}/code'
DATA_DIR = f'{PROJECT_ROOT}/data'
OUTPUT_DIR = f'{PROJECT_ROOT}/outputs'
os.chdir(CODE_DIR)
print(f'Working dir: {os.getcwd()}')

## 2. Metric Collector

`collect_metrics(pred_path)` loads a `test_predictions_*.json` and runs:
* `evaluate_health.compute_health_metrics` -- sugar/sodium delta + satisfaction rate
* `evaluate_flavor.evaluate_flavor` -- I-F cosine on hub-only pairs
* `evaluate_id_ood.evaluate_id_ood` -- MRR split by whether (s,y) was in train

Returns a flat dict of metrics. Stdout from the evaluators is silenced.

In [ ]:
import json
import contextlib
import io

from evaluate_health import compute_health_metrics
from evaluate_flavor import evaluate_flavor
from evaluate_id_ood import evaluate_id_ood


def _quiet(fn, *args, **kw):
    with contextlib.redirect_stdout(io.StringIO()):
        return fn(*args, **kw)


def collect_metrics(pred_path, data_dir=None):
    data_dir = data_dir or DATA_DIR
    if not os.path.exists(pred_path):
        return None
    with open(pred_path) as f:
        pred = json.load(f)
    out = {
        'MRR':    pred['metrics']['MRR'],
        'Hit@1':  pred['metrics']['Hit@1'],
        'Hit@3':  pred['metrics']['Hit@3'],
        'Hit@10': pred['metrics']['Hit@10'],
    }
    try:
        h = _quiet(compute_health_metrics,
                   pred_path, f'{data_dir}/usda_mapping.json', save_to=None)
        out['sugar_sat']   = h['pred']['sugar_g']['satisfaction_rate']
        out['sodium_sat']  = h['pred']['sodium_mg']['satisfaction_rate']
        out['d_sugar_g']   = h['pred']['sugar_g']['avg_delta']
        out['d_sodium_mg'] = h['pred']['sodium_mg']['avg_delta']
    except Exception as e:
        print(f'  [health failed] {pred_path}: {e}')
    try:
        fv = _quiet(evaluate_flavor,
                    pred_path, f'{data_dir}/flavorgraph_edges.csv', save_to=None)
        out['flavor_cos'] = fv['pred_cosine']['mean']
    except Exception as e:
        print(f'  [flavor failed] {pred_path}: {e}')
    try:
        idood = _quiet(evaluate_id_ood,
                        pred_path, f'{data_dir}/pairs_train.csv', save_to=None)
        out['ID_MRR']  = idood['id']['MRR']
        out['OOD_MRR'] = idood['ood']['MRR']
    except Exception as e:
        print(f'  [idood failed] {pred_path}: {e}')
    return out

## 3. Config -- file paths and best lambda

After running training/sweep, edit `BEST_LAMBDA` to the value selected from the sweep,
and update the dicts if your output paths differ.

In [ ]:
# Best lambda chosen from the sweep (look at sweep_summary_v3.csv to decide)
BEST_LAMBDA = 1.0
BEST_TAG = str(BEST_LAMBDA).replace('.', '_')  # filesystem-safe representation

# === Table 1: Main Results (no-train baselines + our methods) ===
MAIN_RESULTS = {
    # No-train baselines
    'Vanilla GISMo':       f'{OUTPUT_DIR}/baseline/test_predictions_baseline.json',
    'GISMo + Hard Filter': f'{OUTPUT_DIR}/filter_baseline/test_predictions_filter_hard_auto.json',
    'GISMo + Soft a=0.5':  f'{OUTPUT_DIR}/filter_baseline/test_predictions_filter_soft_a0.5_auto.json',
    'GISMo + Soft a=1.0':  f'{OUTPUT_DIR}/filter_baseline/test_predictions_filter_soft_a1_auto.json',
    # Our trained methods (at best lambda)
    'Ours v2 (encoder)':   f'{OUTPUT_DIR}/v2_lam{BEST_TAG}/test_predictions_v2_auto.json',
    'Ours v3 (hub)':       f'{OUTPUT_DIR}/v3_lam{BEST_TAG}/test_predictions_v3_auto.json',
    'Ours v4 (decoder)':   f'{OUTPUT_DIR}/v4_lam{BEST_TAG}/test_predictions_v4_auto.json',
}

# === Table 2: Ablation Study (each row drops one component from Full) ===
ABLATIONS = {
    'Full (v3 lambda=best)':       f'{OUTPUT_DIR}/v3_lam{BEST_TAG}/test_predictions_v3_auto.json',
    'w/o nutrition inject (v1 MVP)': f'{OUTPUT_DIR}/v1mvp/test_predictions_mvp_auto.json',
    'w/o L_health (v3 lambda=0)':  f'{OUTPUT_DIR}/v3_lam0/test_predictions_v3_auto.json',
    'w/o flavor compound':         f'{OUTPUT_DIR}/v3_no_compound/test_predictions_v3_auto.json',
}

## 4. Table 1 -- Main Results

Reads each row's predictions, computes all metrics, prints a single comparison table.
Missing files are skipped with a warning so the table still renders with whatever you have.

In [ ]:
import pandas as pd


def build_table(config_dict):
    rows = []
    for label, path in config_dict.items():
        m = collect_metrics(path)
        if m is None:
            print(f'  skip [{label}]: file not found  {path}')
            continue
        rows.append({'row': label, **m})
    if not rows:
        return None
    df = pd.DataFrame(rows).set_index('row')
    return df


def pretty(df, cols=None):
    if df is None:
        print('(no rows to display)')
        return
    if cols is None:
        cols = ['MRR', 'Hit@10', 'sugar_sat', 'sodium_sat',
                'flavor_cos', 'ID_MRR', 'OOD_MRR']
    use = [c for c in cols if c in df.columns]
    print(df[use].round(2).to_string())


print('=== Table 1: Main Results ===')
df_main = build_table(MAIN_RESULTS)
pretty(df_main)
if df_main is not None:
    df_main.to_csv(f'{OUTPUT_DIR}/table1_main_results.csv')
    print(f'\n[saved] {OUTPUT_DIR}/table1_main_results.csv')

## 5. Table 2 -- Ablation Study

Each ablation row drops one design component from the Full model. Reading along the rows:
* **Full vs w/o nutrition inject**: does the v2/v3/v4 architecture beat plain g+L_health (v1 MVP)?
* **Full vs w/o L_health**: does the supervision matter on top of the architecture? (lambda=0 row of the sweep.)
* **Full vs w/o flavor compound**: do I-F edges contribute?

In [ ]:
print('=== Table 2: Ablation Study ===')
df_abl = build_table(ABLATIONS)
pretty(df_abl)
if df_abl is not None:
    df_abl.to_csv(f'{OUTPUT_DIR}/table2_ablation.csv')
    print(f'\n[saved] {OUTPUT_DIR}/table2_ablation.csv')

# Delta-from-Full view for quick reading
if df_abl is not None and 'Full (v3 lambda=best)' in df_abl.index:
    base = df_abl.loc['Full (v3 lambda=best)']
    delta_cols = [c for c in ['MRR', 'Hit@10', 'sugar_sat', 'sodium_sat', 'flavor_cos']
                  if c in df_abl.columns]
    delta = (df_abl[delta_cols] - base[delta_cols]).round(2)
    print('\n--- Delta from Full (component contribution; negative = removing the component hurts) ---')
    print(delta.drop(index='Full (v3 lambda=best)').to_string())

## 6. Lambda Sweep (Pareto curve)

Reads `sweep_summary_v3.csv` produced by `run_lambda_sweep.py`. Prints the trade-off table
and a quick scatter of MRR vs sugar_sat.

In [ ]:
import matplotlib.pyplot as plt

sweep_csv = f'{OUTPUT_DIR}/sweep_summary_v3.csv'
if os.path.exists(sweep_csv):
    df_sw = pd.read_csv(sweep_csv).sort_values('lambda_h').reset_index(drop=True)
    show = ['lambda_h', 'MRR', 'Hit@10', 'sugar_sat_pct',
            'sodium_sat_pct', 'flavor_cos_mean']
    print(df_sw[[c for c in show if c in df_sw.columns]].round(2).to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(df_sw['MRR'], df_sw['sugar_sat_pct'], marker='o')
    for _, r in df_sw.iterrows():
        axes[0].annotate(f"{r['lambda_h']:g}",
                          (r['MRR'], r['sugar_sat_pct']),
                          textcoords='offset points', xytext=(5, 5))
    axes[0].set_xlabel('MRR')
    axes[0].set_ylabel('sugar satisfaction (%)')
    axes[0].set_title('Sugar Pareto (v3)')
    axes[0].grid(alpha=0.3)

    axes[1].plot(df_sw['MRR'], df_sw['sodium_sat_pct'], marker='o', color='C1')
    for _, r in df_sw.iterrows():
        axes[1].annotate(f"{r['lambda_h']:g}",
                          (r['MRR'], r['sodium_sat_pct']),
                          textcoords='offset points', xytext=(5, 5))
    axes[1].set_xlabel('MRR')
    axes[1].set_ylabel('sodium satisfaction (%)')
    axes[1].set_title('Sodium Pareto (v3)')
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/pareto_v3.png', dpi=120)
    plt.show()
    print(f'[saved] {OUTPUT_DIR}/pareto_v3.png')
else:
    print(f'(skip) sweep summary not found: {sweep_csv}')
    print('  run: python run_lambda_sweep.py --variant v3 ...')

## 7. g-override Check -- does the model actually use g?

If the model meaningfully uses the goal vector g, switching `auto -> 1_0 -> 0_1 -> 1_1`
at test time should change the prediction distribution: g=[1,0] (low-sugar goal) should
yield higher sugar_sat than g=[0,1] (low-sodium goal), and vice versa for sodium.

Near-identical numbers across overrides = model is ignoring g.

In [ ]:
g_check_variant = 'v3'
g_check_lam = BEST_TAG

g_configs = {
    'auto': f'{OUTPUT_DIR}/{g_check_variant}_lam{g_check_lam}/test_predictions_{g_check_variant}_auto.json',
    '1_0 (low sugar)':  f'{OUTPUT_DIR}/{g_check_variant}_lam{g_check_lam}/test_predictions_{g_check_variant}_1_0.json',
    '0_1 (low sodium)': f'{OUTPUT_DIR}/{g_check_variant}_lam{g_check_lam}/test_predictions_{g_check_variant}_0_1.json',
    '1_1 (both)':       f'{OUTPUT_DIR}/{g_check_variant}_lam{g_check_lam}/test_predictions_{g_check_variant}_1_1.json',
}

print(f'=== g-override check: {g_check_variant} at lambda={BEST_LAMBDA} ===')
df_g = build_table(g_configs)
pretty(df_g)

if df_g is not None and 'sugar_sat' in df_g.columns and '1_0 (low sugar)' in df_g.index:
    s10 = df_g.loc['1_0 (low sugar)', 'sugar_sat']
    s01 = df_g.loc['0_1 (low sodium)', 'sugar_sat']
    n10 = df_g.loc['1_0 (low sugar)', 'sodium_sat']
    n01 = df_g.loc['0_1 (low sodium)', 'sodium_sat']
    print(f'\n  sugar_sat:  g=1_0 -> {s10:.1f},  g=0_1 -> {s01:.1f}  (diff {s10-s01:+.1f}pp; positive = g matters)')
    print(f'  sodium_sat: g=1_0 -> {n10:.1f},  g=0_1 -> {n01:.1f}  (diff {n01-n10:+.1f}pp; positive = g matters)')

## 8. Case Study

Compare predictions across g overrides for the same (source, recipe). If the model uses g, the
predicted top-1 should differ between g=[1,0] and g=[0,1] for the same query.

In [ ]:
import random

case_variant = 'v3'
case_paths = {
    label: f'{OUTPUT_DIR}/{case_variant}_lam{BEST_TAG}/test_predictions_{case_variant}_{tag}.json'
    for label, tag in [('auto', 'auto'), ('low_sugar', '1_0'),
                        ('low_sodium', '0_1'), ('both', '1_1')]
}

preds = {}
for label, p in case_paths.items():
    if os.path.exists(p):
        with open(p) as f:
            preds[label] = json.load(f)

if not preds:
    print('(no per-g prediction files found; run training with --test_g_overrides auto 1_0 0_1 1_1)')
else:
    # Use the first available file's index order; same row index = same (source, recipe).
    name_df = pd.read_csv(f'{DATA_DIR}/nodes_filtered.csv')
    id2name = dict(zip(name_df['node_id'].astype(int), name_df['name']))
    with open(f'{DATA_DIR}/usda_mapping.json') as f:
        usda = {int(k): v for k, v in json.load(f).items()}

    any_pred = next(iter(preds.values()))
    n = len(any_pred['sources'])
    random.seed(0)
    idx_sample = random.sample(range(n), min(10, n))

    for i in idx_sample:
        s = any_pred['sources'][i]
        y = any_pred['targets'][i]
        print(f"\nsource = {id2name.get(s, str(s))}   |   gt = {id2name.get(y, str(y))}")
        for label, pred in preds.items():
            p = pred['top1'][i]
            line = f'  g={label:<11s} -> {id2name.get(p, str(p))}'
            if s in usda and p in usda:
                d_sugar = float(usda[s].get('sugar_g') or 0) - float(usda[p].get('sugar_g') or 0)
                d_sodium = float(usda[s].get('sodium_mg') or 0) - float(usda[p].get('sodium_mg') or 0)
                line += f'  (d_sugar={d_sugar:+.2f}g, d_sodium={d_sodium:+.0f}mg)'
            print(line)